# Lab 03 · Pandas for Data Manipulation and Cleaning
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais

**CLO 6** (C-4 Analyze, PLO 2 Problem Analysis): *Analyze real-world datasets by applying data manipulation and visualization techniques.*

**Data:** `pakistan_used_cars.csv`, 24,973 real used-car ads from OLX Pakistan (Open Data Pakistan, *Pakistan Used Cars*, 2021). The ads are real and contain the problems real data has. Two kinds of formatting problem were added for practice (some prices written as text such as `Rs 1,430,000`, some city names in the wrong case or with extra spaces).

Name: Syed Abdulrehman  ·  Roll No: 2024_SE_18

## Warm-up (not graded): the class example from the lecture
Run this cell and repeat any lecture example on this small table before you start the tasks. `clean` is the tidy version, `messy` has the six kinds of problem.

In [13]:
import numpy as np
import pandas as pd
pd.set_option('display.width', 140)

clean = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Zainab"],
    "City": ["Muzaffarabad", "Mirpur", "Bagh", "Muzaffarabad", "Rawalakot", "Mirpur", "Kotli", "Muzaffarabad"],
    "Maths": [78, 91, 45, 85, 62, 73, 38, 88],
    "Physics": [65, 88, 52, 79, 48, 81, 41, 92],
    "Attendance": [92, 97, 70, 88, 75, 90, 60, 95]})

messy = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-04", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Fatima", "Zainab"],
    "City": [" Muzaffarabad", "Mirpur", "Bagh", "MUZAFFARABAD", "Rawalakot", "Mirpur", "Kotli", "MUZAFFARABAD", "Muzafarabad"],
    "Maths": [78, 91, np.nan, 85, 62, 73, 380, 85, 88],
    "Physics": [65, 88, 52, 79, 48, 105, 41, 79, 92],
    "Attendance": ["92%", "97", "70", "88", "75%", "90", "60", "88", "95"]})
print(clean)

    Roll    Name          City  Maths  Physics  Attendance
0  SE-01     Ali  Muzaffarabad     78       65          92
1  SE-02  Ayesha        Mirpur     91       88          97
2  SE-03   Bilal          Bagh     45       52          70
3  SE-04  Fatima  Muzaffarabad     85       79          88
4  SE-05   Hamza     Rawalakot     62       48          75
5  SE-06    Sana        Mirpur     73       81          90
6  SE-07   Usman         Kotli     38       41          60
7  SE-08  Zainab  Muzaffarabad     88       92          95


In [14]:
# Warm-up (not graded): lecture examples on the small class table
print(clean[clean['Maths'] > 80])                                   # one condition
print(clean[(clean['Maths'] > 80) & (clean['City'] == 'Mirpur')])   # two conditions need & and brackets
print("messy Maths mean (dirty):", messy['Maths'].mean())            # 117.75, wrong because of 380 and a duplicate

    Roll    Name          City  Maths  Physics  Attendance
1  SE-02  Ayesha        Mirpur     91       88          97
3  SE-04  Fatima  Muzaffarabad     85       79          88
7  SE-08  Zainab  Muzaffarabad     88       92          95
    Roll    Name    City  Maths  Physics  Attendance
1  SE-02  Ayesha  Mirpur     91       88          97
messy Maths mean (dirty): 117.75


### Setup
Put `pakistan_used_cars.csv` in the same folder as this notebook. In Colab: click the folder icon on the left → Upload → choose the CSV.

## Task 1 · Load and first look
- **(a)** Read `pakistan_used_cars.csv` into `df` and print its shape.
- **(b)** Show the first 5 rows.
- **(c)** Print the column names.
- **(d)** Show the data type of every column.
- **(e)** Which column should be a number but is not? Show two of its values that explain why.

In [15]:
# T1: Load and first look
df = pd.read_csv("pakistan_used_cars.csv")
print("T1a: shape =", df.shape)                                   # (a)

print("T1b: first 5 rows")                                        # (b)
print(df.head())

print("T1c: column names =", list(df.columns))                    # (c)

print("T1d: data types")                                          # (d)
print(df.dtypes)

# (e) Price should be a number but is text: some values are written like "Rs 435,000"
not_numeric = pd.to_numeric(df["Price"], errors="coerce").isna()  # True where Price cannot be read as a number
print("T1e: Price is the column that should be a number but is text.")
print("T1e: two values that explain why =", df.loc[not_numeric, "Price"].head(2).tolist())
print("T1e: number of Price values written as text with Rs / commas =", not_numeric.sum())

T1a: shape = (24973, 9)
T1b: first 5 rows
    Brand Condition    Fuel  KMs Driven        Model    Price Registered City Transaction Type    Year
0  Toyota      Used  Diesel         1.0        Prado  2100000         Karachi             Cash  1997.0
1  Suzuki      Used  Petrol    100000.0        Bolan   380000         Karachi             Cash  2006.0
2  Suzuki      Used     CNG     12345.0        Bolan   340000         Karachi             Cash  1998.0
3  Suzuki      Used  Petrol     94000.0         Alto   535000         Karachi             Cash  2010.0
4  Toyota      Used  Petrol    100000.0  Corolla XLI  1430000         Karachi             Cash  2013.0
T1c: column names = ['Brand', 'Condition', 'Fuel', 'KMs Driven', 'Model', 'Price', 'Registered City', 'Transaction Type', 'Year']
T1d: data types
Brand                object
Condition            object
Fuel                 object
KMs Driven          float64
Model                object
Price                object
Registered City      objec

## Task 2 · Retrieve data
- **(a)** The `Brand` column (first 5 values).
- **(b)** The columns `Brand`, `Model`, `Price` together (first 5 rows).
- **(c)** The first row, using `iloc`.
- **(d)** The `Model` in the row with index 100, using `loc`.
- **(e)** How many cars are registered in `"Islamabad"`?
- **(f)** How many **Honda** cars run on **Hybrid** fuel?
- **(g)** Cars from 2018 or newer, newest first (Brand, Model, Year, Price; first 10).
- **(h)** Try `df[df["Price"] > 1_000_000]`. What happens and why? Answer in a Markdown cell.

In [16]:
# T2: Retrieve data (on the raw, not-yet-cleaned table)
print("T2a: Brand (first 5):")                                         # (a)
print(df["Brand"].head())

print("T2b: Brand, Model, Price (first 5 rows):")                      # (b) double brackets -> a DataFrame
print(df[["Brand", "Model", "Price"]].head())

print("T2c: first row with iloc:")                                    # (c) iloc = by position
print(df.iloc[0])

print("T2d: Model in the row with index label 100 =", df.loc[100, "Model"])   # (d) loc = by label

print("T2e: cars registered in exactly 'Islamabad' (raw, exact match) =",
      (df["Registered City"] == "Islamabad").sum())                   # (e)
print("T2e: same count after ignoring extra spaces / capitals (what Task 3 will fix) =",
      (df["Registered City"].str.strip().str.title() == "Islamabad").sum())

print("T2f: Honda cars on Hybrid fuel =",
      ((df["Brand"] == "Honda") & (df["Fuel"] == "Hybrid")).sum())    # (f) two conditions: & with brackets

print("T2g: cars from 2018 or newer, newest first (first 10):")        # (g)
print(df[df["Year"] >= 2018]
      .sort_values("Year", ascending=False, kind="stable")[["Brand", "Model", "Year", "Price"]]
      .head(10))

# (h) try to filter the raw Price column with a number
try:
    df[df["Price"] > 1_000_000]
except TypeError as e:
    print("T2h: TypeError:", e)

T2a: Brand (first 5):
0    Toyota
1    Suzuki
2    Suzuki
3    Suzuki
4    Toyota
Name: Brand, dtype: object
T2b: Brand, Model, Price (first 5 rows):
    Brand        Model    Price
0  Toyota        Prado  2100000
1  Suzuki        Bolan   380000
2  Suzuki        Bolan   340000
3  Suzuki         Alto   535000
4  Toyota  Corolla XLI  1430000
T2c: first row with iloc:
Brand                Toyota
Condition              Used
Fuel                 Diesel
KMs Driven              1.0
Model                 Prado
Price               2100000
Registered City     Karachi
Transaction Type       Cash
Year                 1997.0
Name: 0, dtype: object
T2d: Model in the row with index label 100 = Mehran VX
T2e: cars registered in exactly 'Islamabad' (raw, exact match) = 958
T2e: same count after ignoring extra spaces / capitals (what Task 3 will fix) = 986
T2f: Honda cars on Hybrid fuel = 202
T2g: cars from 2018 or newer, newest first (first 10):
        Brand                      Model    Year       Pr

**Answer (T2h):** The line fails with `TypeError: '>' not supported between instances of 'str' and 'int'`. The `Price` column is stored as text (`object`/`str` type) because some prices are written like `"Rs 435,000"`, and one text value makes the whole column text. Pandas cannot compare text with the number `1_000_000`. The fix is Task 3(b) and 3(c): remove `"Rs"` and the commas, then convert `Price` to `int64`.

## Task 3 · Clean the data: the six types of problem
Fix the problems **in this order** (text first, then duplicates):
- **(a) Missing values:** count them per column. Drop ads with no `Brand` or no `Model`; fill missing `Registered City`, `Fuel` and `Transaction Type` with `"Unknown"`; leave `Year` and `KMs Driven` as NaN. Print how many rows were removed.
- **(b) Formatting:** remove extra spaces and fix the case of city names (`.str.strip().str.title()`); remove `"Rs"` and commas from `Price`.
- **(c) Data types:** convert `Price` to `int64` and `Year` to `Int64`.
- **(d) Inconsistent categories:** `"Sukkar"` → `"Sukkur"` and `"Corrolla Altis"` → `"Corolla Altis"`. Print the number of different cities before and after.
- **(e) Duplicates:** count exact duplicate ads, drop them, print the new row count.
- **(f) Invalid values:** set `KMs Driven` to NaN when it is below 100, above 1,000,000, or the placeholder 12345 / 123456. Drop ads with the fake prices 12,345,678 or 87,654,321, and Suzuki/Daihatsu ads above 5,000,000.
- **(g) Self-check:** your final shape must be **(20484, 9)**.

In [17]:
# T3: Clean the data, in the required order
# ---- (a) Missing values -------------------------------------------------------------
print("T3a: missing values per column (before):")
print(df.isna().sum())
rows_before = len(df)
df = df.dropna(subset=["Brand", "Model"])                              # useless without brand or model
df = df.fillna({"Registered City": "Unknown", "Fuel": "Unknown", "Transaction Type": "Unknown"})   # text labels -> "Unknown"
# Year and KMs Driven are left as NaN on purpose (we do not know the true value)
print("T3a: rows removed (no Brand or no Model) =", rows_before - len(df), "| rows left =", len(df))

# ---- (b) Formatting ------------------------------------------------------------------
df["Registered City"] = df["Registered City"].str.strip().str.title()
df["Price"] = df["Price"].str.replace("Rs", "", regex=False).str.replace(",", "", regex=False).str.strip()
print("T3b: city spaces/case fixed, 'Rs' and commas removed. Example Price values:", df["Price"].head(3).tolist())

# ---- (c) Data types ------------------------------------------------------------------
df["Price"] = df["Price"].astype("int64")
df["Year"] = df["Year"].astype("Int64")                                # whole numbers that allow NaN
print("T3c: Price dtype =", df["Price"].dtype, "| Year dtype =", df["Year"].dtype)

# ---- (d) Inconsistent categories ----------------------------------------------------
print("T3d: different cities BEFORE =", df["Registered City"].nunique())
df["Registered City"] = df["Registered City"].replace({"Sukkar": "Sukkur"})
df["Model"] = df["Model"].replace({"Corrolla Altis": "Corolla Altis"})
print("T3d: different cities AFTER  =", df["Registered City"].nunique())

# ---- (e) Duplicates ------------------------------------------------------------------
print("T3e: exact duplicate ads =", df.duplicated().sum())
df = df.drop_duplicates()                                              # assign back!
print("T3e: rows after dropping duplicates =", len(df))

# ---- (f) Invalid values --------------------------------------------------------------
print("T3f: mean KMs Driven BEFORE the invalid-value rule =", round(df["KMs Driven"].mean()), "km")
bad_km = ((df["KMs Driven"] < 100) | (df["KMs Driven"] > 1_000_000)
          | df["KMs Driven"].isin([12345, 123456]))                    # impossible or placeholder mileage
df.loc[bad_km, "KMs Driven"] = np.nan                                  # set to NaN, do not guess
print("T3f: KMs Driven set to NaN =", bad_km.sum(), "values")
print("T3f: mean KMs Driven AFTER the rule =", round(df["KMs Driven"].mean()), "km")

fake_price = df["Price"].isin([12_345_678, 87_654_321])                # placeholder prices
print("T3f: ads with fake prices dropped =", fake_price.sum())
df = df[~fake_price]

too_dear = df["Brand"].isin(["Suzuki", "Daihatsu"]) & (df["Price"] > 5_000_000)
print("T3f: Suzuki/Daihatsu ads above Rs 5,000,000 dropped =", too_dear.sum())
df = df[~too_dear]

# ---- (g) Self-check ------------------------------------------------------------------
print("T3g: final shape =", df.shape, "| expected (20484, 9) | match =", df.shape == (20484, 9))

T3a: missing values per column (before):
Brand               2137
Condition           2136
Fuel                2445
KMs Driven          2286
Model               2448
Price                  0
Registered City     4636
Transaction Type    2445
Year                2284
dtype: int64
T3a: rows removed (no Brand or no Model) = 2448 | rows left = 22525
T3b: city spaces/case fixed, 'Rs' and commas removed. Example Price values: ['2100000', '380000', '340000']
T3c: Price dtype = int64 | Year dtype = Int64
T3d: different cities BEFORE = 62
T3d: different cities AFTER  = 61
T3e: exact duplicate ads = 2028
T3e: rows after dropping duplicates = 20497
T3f: mean KMs Driven BEFORE the invalid-value rule = 131160 km
T3f: KMs Driven set to NaN = 2852 values
T3f: mean KMs Driven AFTER the rule = 93238 km
T3f: ads with fake prices dropped = 3
T3f: Suzuki/Daihatsu ads above Rs 5,000,000 dropped = 10
T3g: final shape = (20484, 9) | expected (20484, 9) | match = True


## Task 4 · Analyse
- **(a)** New column `Price_lakh` = price in lakh rupees (÷ 100,000).
- **(b)** New column `Age` = 2020 − `Year`.
- **(c)** The 5 most common brands.
- **(d)** The 5 cities with the most ads.
- **(e)** Median `Price_lakh` for each fuel type, highest first.
- **(f)** Median `Price_lakh` of a **Corolla GLI** for each year 2010–2018. How much value does it lose per year?
- **(g)** The 5 most expensive ads. Does one of them look wrong? Why?
- **(h)** Number of ads and median price for Cash vs Installment/Leasing. Interpret in 1–2 lines.

In [18]:
# T4: Analyse
df["Price_lakh"] = df["Price"] / 100_000                                # (a) 1 lakh = 100,000 rupees
print("T4a: Price_lakh (first 3):", df["Price_lakh"].head(3).tolist())

df["Age"] = 2020 - df["Year"]                                           # (b)
print("T4b: Age (first 3, years):", df["Age"].head(3).tolist())

print("T4c: 5 most common brands (number of ads):")                     # (c)
print(df["Brand"].value_counts().head(5))

print("T4d: 5 cities with the most ads (Unknown = city not given in the ad):")   # (d)
print(df["Registered City"].value_counts().head(5))
print("T4d: same, leaving out 'Unknown':")
print(df.loc[df["Registered City"] != "Unknown", "Registered City"].value_counts().head(5))

print("T4e: median price (lakh Rs) per fuel type, highest first:")      # (e)
print(df.groupby("Fuel")["Price_lakh"].median().sort_values(ascending=False))

gli = df[(df["Model"] == "Corolla GLI") & df["Year"].between(2010, 2018)]   # (f)
gli_median = gli.groupby("Year")["Price_lakh"].median()
print("T4f: Corolla GLI median price (lakh Rs) per model year:")
print(gli_median)
loss_per_year = (gli_median.loc[2018] - gli_median.loc[2010]) / (2018 - 2010)
print("T4f: average value lost per year of age (2010 to 2018) =", round(loss_per_year, 2), "lakh Rs")
slope = np.polyfit(gli_median.index.astype(int), gli_median.values, 1)[0]
print("T4f: same estimate from a straight-line fit =", round(slope, 2), "lakh Rs per year")

print("T4g: 5 most expensive ads:")                                     # (g)
print(df.nlargest(5, "Price")[["Brand", "Model", "Year", "KMs Driven", "Condition", "Price", "Price_lakh"]])
landcruiser = df[df["Model"] == "Land Cruiser"]["Price_lakh"]
print("T4g: Land Cruiser ads =", len(landcruiser), "| median price =", landcruiser.median(), "lakh Rs",
      "| maximum price =", landcruiser.max(), "lakh Rs")

print("T4h: Cash vs Installment/Leasing: number of ads and median price (lakh Rs):")   # (h)
print(df.groupby("Transaction Type")["Price_lakh"].agg(ads="count", median_price_lakh="median"))
print("T4h extra: median model year per transaction type:")
print(df.groupby("Transaction Type")["Year"].median())

T4a: Price_lakh (first 3): [21.0, 3.8, 3.4]
T4b: Age (first 3, years): [23, 14, 22]
T4c: 5 most common brands (number of ads):
Brand
Suzuki      8621
Toyota      4535
Honda       3025
Daihatsu    2231
Nissan       694
Name: count, dtype: int64
T4d: 5 cities with the most ads (Unknown = city not given in the ad):
Registered City
Karachi       15461
Unknown        2040
Lahore         1601
Islamabad       777
Rawalpindi      119
Name: count, dtype: int64
T4d: same, leaving out 'Unknown':
Registered City
Karachi       15461
Lahore         1601
Islamabad       777
Rawalpindi      119
Multan           88
Name: count, dtype: int64
T4e: median price (lakh Rs) per fuel type, highest first:
Fuel
Hybrid    16.2750
Diesel    13.0000
Petrol     8.7525
CNG        3.8500
LPG        1.5750
Name: Price_lakh, dtype: float64
T4f: Corolla GLI median price (lakh Rs) per model year:
Year
2010    12.250
2011    13.325
2012    14.125
2013    14.950
2014    16.750
2015    17.900
2016    19.000
2017    21.000
2

**Interpretation (T4f):** A Corolla GLI loses value steadily with age: the median price rises from 12.25 lakh for a 2010 model to 23.20 lakh for a 2018 model. That is about **1.3 to 1.4 lakh rupees (roughly Rs 133,000 to 137,000) lost for every extra year of age** (1.37 from the 2010 and 2018 medians, 1.33 from a straight-line fit through all nine years), so a 2010 car costs about half of a 2018 one.

**Interpretation (T4g):** Yes, the most expensive ad looks wrong. It is a 2017 Toyota Land Cruiser listed at Rs 67,000,000 (670 lakh), while the median Land Cruiser ad is only 24.0 lakh, so this price is about 28 times the typical price. It is also marked **"New"** while showing 32,222 km, and the repeating digits (32222, and 22222 and 3222 in the 3rd and 4th ads) look like typing shortcuts. It is possible but unbelievable, so it is an outlier: I flag it and do not delete it automatically, because the lab says to check an outlier first (for example, by looking at the ad).

**Interpretation (T4h):** Only 850 ads (about 4 %) are Installment/Leasing against 19,634 Cash ads, and the median Installment price is 3.0 lakh versus 6.9 lakh for Cash. That is surprising, because the Installment cars are newer (median model year 2016 versus 2007). So the price in an installment ad is probably the down-payment or advance, not the full price of the car. This is my explanation, not something the data proves, so the two groups should not be compared directly.

## Task 5 · Save your work
Save the cleaned DataFrame as `pakistan_used_cars_clean.csv` (Lab 04 uses it) and print its shape.

In [19]:
# T5: Save your work
df.to_csv("pakistan_used_cars_clean.csv", index=False)                 # index=False: do not save the row numbers
saved = pd.read_csv("pakistan_used_cars_clean.csv")                    # read it back to confirm
print("T5: saved pakistan_used_cars_clean.csv | shape of DataFrame =", df.shape, "| shape read back =", saved.shape)

T5: saved pakistan_used_cars_clean.csv | shape of DataFrame = (20484, 11) | shape read back = (20484, 11)


## Task 6 · Reflection (Markdown cell)
Write 3–5 sentences: which of the six cleaning problems would have given the most misleading answer if you had ignored it, and two things you learned about the used-car market in Pakistan.

**Reflection:** The problem that would have misled me most is **invalid values**: 2,852 mileage entries were impossible placeholders such as 12345 or 1 km, and ignoring them gave an average of 131,160 km instead of 93,238 km, with no error message at all. Duplicates came second, because 2,028 repeated ads (about 9 % of the 22,525 rows) would have been counted twice in every average and count. About the car market, I learned that hybrids have the highest median price (16.28 lakh) and CNG cars are cheap (3.85 lakh), and that a Corolla GLI loses about 1.3 to 1.4 lakh rupees every year. I also learned that Karachi has about 75 % of all ads (15,461 of 20,484).